## Load unique s1-v4 river endpoints for the shared S04/S10 900 m grid


In [ ]:
# Load s1-v4 endpoints for the configured S04/S10 900 m grid.
import os
import pandas as pd
import pyproj
from pyproj import Transformer

site_selection_folder = './site_selection'
csv_path = os.path.join(site_selection_folder, 'configured_site_river_segment_endpoints.csv')
raw_points = pd.read_csv(csv_path)
required_columns = {
    'site_id', 'raster_row', 'raster_col', 'point_id', 'endpoint_type',
    'easting', 'northing', 'longitude', 'latitude'
}
missing = required_columns.difference(raw_points.columns)
if missing:
    raise ValueError(f's1-v4 endpoint CSV is missing columns: {sorted(missing)}')

print(f'Loaded {len(raw_points)} rows from {csv_path}')
cell_pairs = raw_points[['raster_row', 'raster_col']].drop_duplicates()
if len(cell_pairs) != 1:
    raise ValueError(f'Expected one 900 m cell, found: {cell_pairs.to_dict("records")}')
grid_row, grid_col = cell_pairs.iloc[0].astype(int)
grid_id = f'cell_r{grid_row:03d}_c{grid_col:03d}'
print(f'Using shared 900 m grid: {grid_id}')

# S04 and S10 share this grid, so retain each physical endpoint once.
grouped = raw_points.groupby(['easting', 'northing'], sort=True, as_index=False).agg(
    endpoint_type=('endpoint_type', 'first'),
    longitude=('longitude', 'first'),
    latitude=('latitude', 'first'),
    source_site_ids=('site_id', lambda values: ';'.join(sorted(set(values)))),
)
if len(grouped) != 48:
    raise ValueError(f'Expected 48 unique endpoints after S04/S10 de-duplication, found {len(grouped)}')

grouped.insert(0, 'point_id', range(1, len(grouped) + 1))
grouped.insert(0, 'grid_id', grid_id)
grouped.insert(0, 'raster_col', grid_col)
grouped.insert(0, 'raster_row', grid_row)
grouped['site_name'] = grouped['grid_id']

# Convert endpoint longitude/latitude to the DayMet CRS used by the ATS mesh.
crs_daymet_pyproj = pyproj.CRS.from_proj4(
    '+proj=lcc +lat_1=25 +lat_2=60 +lat_0=42.5 +lon_0=-100 '
    '+x_0=0 +y_0=0 +ellps=WGS84 +units=m +no_defs'
)
to_daymet = Transformer.from_crs('EPSG:4326', crs_daymet_pyproj, always_xy=True)
grouped['x_daymet'], grouped['y_daymet'] = to_daymet.transform(
    grouped['longitude'].to_numpy(), grouped['latitude'].to_numpy()
)

df_points = grouped[[
    'grid_id', 'raster_row', 'raster_col', 'point_id', 'endpoint_type',
    'source_site_ids', 'easting', 'northing', 'longitude', 'latitude',
    'x_daymet', 'y_daymet', 'site_name'
]].copy()

print(f'De-duplicated {len(raw_points)} CSV rows to {len(df_points)} unique endpoints.')
print(df_points[['grid_id', 'point_id', 'endpoint_type', 'source_site_ids', 'longitude', 'latitude']])


In [ ]:
df_points


## Load river network and 3D ATS surface/subsurface vis files

In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt
import h5py as h5
import matplotlib.collections as mc

import scipy.signal
from datetime import datetime, timedelta
import pandas as pd

import ats_xdmf as xdmf
import time

# Essential imports for watershed workflow
import watershed_workflow
import watershed_workflow.source_list
import watershed_workflow.plot
import pyproj

# Load config
import json
with open('config.json', 'r') as f:
    config = json.load(f)
case_config = config['case']
site_name = case_config['site_name']

# Define the output model directory where ats_vis data (.h5) are located
model_dir = '/global/cfs/cdirs/m1800/naches_run2_share/Naches-2'

# Define the Parameters
rho = 997  # density of water, kg m^-3
g = 9.80665  # gravity, m s^-2
patm = 101325  # atmospheric pressure, Pascals

In [ ]:
## get stream network

# set up a dictionary of source objects
sources = watershed_workflow.source_list.get_default_sources()
sources['hydrography'] = watershed_workflow.source_list.hydrography_sources['NHD Plus']
sources['HUC'] = watershed_workflow.source_list.huc_sources['NHD Plus']
sources['DEM'] = watershed_workflow.source_list.dem_sources['NED 1/3 arc-second']
#sources['geologic structure'] = watershed_workflow.source_list.FileManagerGLHYMPS('/global/cfs/cdirs/m1800/zhi/ww/scripts/data/soil_structure/GLHYMPS/GLHYMPS.shp')
#sources['depth to bedrock'] = watershed_workflow.source_list.FileManagerRaster('/global/cfs/cdirs/m1800/zhi/ww/scripts/data/soil_structure/SoilGrids2017/BDTICM_M_250m_ll.tif')
#sources['geologic structure'] = watershed_workflow.source_list.FileManagerGLHYMPS('/global/cfs/cdirs/m1800/xiaoyi/ARW-ELMPF/data/ww_data_from_zhi/soil_structure/GLHYMPS/GLHYMPS.shp')
#sources['depth to bedrock'] = watershed_workflow.source_list.FileManagerRaster('/global/cfs/cdirs/m1800/xiaoyi/ARW-ELMPF/data/ww_data_from_zhi/soil_structure/SoilGrids2017/BDTICM_M_250m_ll.tif')
watershed_workflow.source_list.log_sources(sources)

# Set up watershed workflow CRS (DayMet CRS)
crs_daymet = watershed_workflow.crs.daymet_crs()

# Set up sources
sources = watershed_workflow.source_list.get_default_sources()
sources['hydrography'] = watershed_workflow.source_list.hydrography_sources['NHD Plus']
sources['HUC'] = watershed_workflow.source_list.huc_sources['NHD Plus']

# Parameters for river extraction
hucs_config = [case_config['hucs']]
ignore_small_rivers = 2
prune_by_area_fraction = 0.0

# Get HUC12 list
def get_huc12(hucs):
    huc12_list = []
    for huc in hucs:
        if len(huc) == 12:
            huc12_list.append(huc)
        elif len(huc) == 10:
            for i in range(1,20):
                huc12_list.append(huc+str(i).zfill(2))
        elif len(huc) == 8:
            for i in range(1,20):
                for j in range(1,20):
                    huc12_list.append(huc+str(i).zfill(2)+str(j).zfill(2))
    return huc12_list

hucs = get_huc12(hucs_config)
huc_level = 12

print(f"Processing HUCs: {hucs[:5]}...")

# Load watershed HUCs
my_hucs = []
for huc in hucs:
    _, ws = watershed_workflow.get_hucs(sources['HUC'], huc, huc_level, crs_daymet)
    my_hucs.extend(ws)

watershed = watershed_workflow.split_hucs.SplitHUCs(my_hucs)

# Download/collect the river network
print("Downloading river network...")
_, reaches = watershed_workflow.get_reaches(sources['hydrography'], hucs[0], 
                                            watershed.exterior(), crs_daymet, crs_daymet,
                                            in_network=True, properties=True)

# Construct river network
rivers = watershed_workflow.construct_rivers(reaches, method='hydroseq',
                                             ignore_small_rivers=ignore_small_rivers,
                                             prune_by_area=prune_by_area_fraction * watershed.exterior().area * 1.e-6,
                                             remove_diversions=True,
                                             remove_braided_divergences=True)

print(f"Number of rivers: {len(rivers)}")

In [ ]:
# Load saved river line segments from river cells
import pickle

with open('./site_selection/river_cells_and_segments.pkl', 'rb') as f:
    river_data = pickle.load(f)

line_segs_in_cells = river_data['line_segments_in_cells']  # 2-point LineStrings in DayMet CRS
crs_daymet_loaded = river_data['crs']

print(f"Loaded {len(line_segs_in_cells)} river line segments from river cells")
print(f"CRS: {river_data['metadata']['crs_name']}")

In [ ]:
# Load ATS 3D simulation results

print("="*80)
print("LOADING ATS 3D SIMULATION RESULTS")
print("="*80)

# Load surface mesh
start = time.time()
visfile_surface = xdmf.VisFile(directory=model_dir,
                               domain="surface", 
                               filename="ats_vis_surface_data.h5", 
                               mesh_filename="ats_vis_surface_mesh.h5")
visfile_surface.loadMesh()
end = time.time()
print(f"Time cost for surface VisFile: {end - start:.6f} seconds")

# Load subsurface mesh
start = time.time()
visfile_subsurface = xdmf.VisFile(directory=model_dir,
                                  filename="ats_vis_data.h5", 
                                  mesh_filename="ats_vis_mesh.h5")
visfile_subsurface.loadMesh(columnar=True)
end = time.time()
print(f"Time cost for subsurface VisFile: {end - start:.6f} seconds")

# Get subsurface pressure
start = time.time()
pressure_subsurface = visfile_subsurface.getArray('pressure')
end = time.time()
print(f"Time cost for getting pressure_subsurface: {end - start:.6f} seconds")
print(f"Pressure subsurface shape: {pressure_subsurface.shape}")

# Function to estimate WTD based on pressure
def get_ats_wtd_pressurebased(pressure_subsurface, visfile_surface, visfile_subsurface):
    iz_coord = visfile_subsurface.centroids[:, :, -1]
    
    # Find Equivalent Surface and Subsurface ID based on cell centroids
    surface_centroids_rounded = np.round(visfile_surface.centroids[:, :2], 4)
    subsurface_centroids_rounded = np.round(visfile_subsurface.centroids[:, 0, :2], 4)
    surface_centroids_expanded = surface_centroids_rounded[:, np.newaxis, :]
    subsurface_centroids_expanded = subsurface_centroids_rounded[np.newaxis, :, :]
    matches = np.all(surface_centroids_expanded == subsurface_centroids_expanded, axis=-1)
    surface_indices, subsurface_indices = np.nonzero(matches)
    surface_subsurface_IDs = np.column_stack((surface_indices, subsurface_indices))
    
    # Estimate WTD based on pressure
    ih = (pressure_subsurface - patm) / (rho * g)
    mask = ih > 0
    first_false_idx = (~mask).argmax(axis=-1)
    max_len = mask.shape[-1]
    indices = np.arange(max_len)
    mask2 = indices < first_false_idx[..., np.newaxis]
    all_true_rows = (first_false_idx == 0)
    mask2[all_true_rows] = True
    sat_idx = mask2[:, :, ::-1].argmax(axis=-1)
    sat_idx = mask2.shape[-1] - 1 - sat_idx
    sat_idx[~mask2.any(axis=-1)] = 0
    
    # WTD elevation
    iH_rev = ih[np.arange(ih.shape[0])[:, None], np.arange(ih.shape[1]), sat_idx] + iz_coord[np.arange(ih.shape[1]), sat_idx]
    first_depth_to_centroid = visfile_surface.centroids[surface_subsurface_IDs[0][0], -1] - visfile_subsurface.centroids[surface_subsurface_IDs[0][1], -1, -1]
    
    # WTD (from surface)
    head_pressure_based = -(iz_coord[:, -1] + first_depth_to_centroid - iH_rev)
    wtdep_pressure_based = np.maximum(iz_coord[:, -1] + first_depth_to_centroid - iH_rev, 0)
    pwdep_pressure_based = -np.minimum(iz_coord[:, -1] + first_depth_to_centroid - iH_rev, 0)
    
    # Re-arrange WTD in accordance to surface IDs
    head_rearranged = head_pressure_based[:, subsurface_indices]
    wtdep_rearranged = wtdep_pressure_based[:, subsurface_indices]
    pwdep_rearranged = pwdep_pressure_based[:, subsurface_indices]
    
    return surface_subsurface_IDs, head_rearranged, wtdep_rearranged, pwdep_rearranged

# Get WTD
start = time.time()
surface_subsurface_IDs, head_rearranged, wtdep_rearranged, pwdep_rearranged = get_ats_wtd_pressurebased(
    pressure_subsurface=pressure_subsurface,
    visfile_surface=visfile_surface, 
    visfile_subsurface=visfile_subsurface)
end = time.time()
print(f"Time cost for get_ats_wtd_pressurebased: {end - start:.6f} seconds")

# Get surface coordinates
surface_x_coord = visfile_surface.centroids[:, 0]
surface_y_coord = visfile_surface.centroids[:, 1]
surface_times = visfile_surface.times
print(f"Number of timesteps: {len(surface_times)}")

## Generate ATS boundary-head plots for all unique endpoints


In [ ]:
# Loop through all points in df_points and create analysis plots
import os

# Ensure the output directory exists
output_dir = './site_selection/s2_checkBChead'
os.makedirs(output_dir, exist_ok=True)

# Define N closest points to analyze
N = 7

print("="*80)
print(f"GENERATING PLOTS FOR ALL {len(df_points)} POINTS")
print("="*80)

for point_idx in range(len(df_points)):
    cur_point = df_points.iloc[point_idx]
    point_coords = (cur_point['x_daymet'], cur_point['y_daymet'])
    
    print(f"\n[{point_idx+1}/{len(df_points)}] Analyzing point: {cur_point['site_name']} P{cur_point['point_id']}")
    print(f"  Coordinates: ({point_coords[0]:.2f}, {point_coords[1]:.2f})")
    
    # Compute Euclidean distances to the point
    dist_point = np.sqrt((surface_x_coord - point_coords[0])**2 + (surface_y_coord - point_coords[1])**2)
    
    # Get the indices of the N closest points
    point_indices = np.argpartition(dist_point, N)[:N]
    point_indices = point_indices[np.argsort(dist_point[point_indices])]
    point_distances = dist_point[point_indices]
    
    print(f"  Closest mesh cell: Index {point_indices[0]}, Distance: {point_distances[0]:.4f} m")
    
    # Extract head and ponded water depth for top N closest points
    point_heads = [head_rearranged[:, idx] for idx in point_indices]
    point_pwdeps = [pwdep_rearranged[:, idx] for idx in point_indices]
    
    # Get surface ponded depth from surface vis file
    surface_ponded_depth = visfile_surface.getArray('surface-ponded_depth')
    closest_pwdep_surface_vis = surface_ponded_depth[:, point_indices[0]]
    
    # Get mesh info for plotting
    etype, coords, conn = xdmf.meshXYZ(model_dir, "ats_vis_surface_mesh.h5")
    polygon_coords = [coords[c][:, :2] for c in conn]
    
    # Create polygons collection
    polygons = mc.PolyCollection(polygon_coords, edgecolor='k', cmap='Blues', linewidth=0.5)
    data = visfile_surface.get('surface-ponded_depth', visfile_surface.cycles[0])
    polygons.set_array(data)
    polygons.set_clim(vmin=0, vmax=0.01)
    
    # Create figure with 4 subplots
    fig = plt.figure(figsize=(16, 12))
    gs = fig.add_gridspec(2, 2, hspace=0.3, wspace=0.3)
    
    # Subplot (0,0): Mesh plot with highlighted closest cell
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.add_collection(polygons)
    
    # Highlight the closest polygon
    closest_idx = point_indices[0]
    highlighted_polygon = mc.PolyCollection([polygon_coords[closest_idx]], 
                                           edgecolor='green', 
                                           facecolors='none', 
                                           linewidth=3)
    ax1.add_collection(highlighted_polygon)
    
    # add rivers
    watershed_workflow.plot.rivers(rivers, crs_daymet, ax=ax1, colors='red', linewidth=2.0)
    
    # Overlay river line segments from river cells in GOLD
    for line_seg in line_segs_in_cells:
        coords_seg = np.array(line_seg.coords)
        ax1.plot(coords_seg[:, 0], coords_seg[:, 1], color='gold', linewidth=3, zorder=10, alpha=0.9)
    
    # Zoom to region around point with buffer
    x_range = 2000  # meters
    y_range = 2000  # meters
    ax1.set_xlim(point_coords[0] - x_range, point_coords[0] + x_range)
    ax1.set_ylim(point_coords[1] - y_range, point_coords[1] + y_range)
    
    ax1.set_aspect('equal')
    ax1.set_xlabel('X [m]', fontsize=11)
    ax1.set_ylabel('Y [m]', fontsize=11)
    cbar1 = plt.colorbar(polygons, ax=ax1, fraction=0.046, pad=0.04)
    cbar1.set_label('surface-ponded_depth [m]', fontsize=10)
    ax1.set_title(f'Surface Mesh - Cycle {visfile_surface.cycles[0]}', fontsize=12, fontweight='bold')
    
    # Plot the point and top N closest centroids
    ax1.plot(*point_coords, 'gx', markersize=12, markeredgewidth=2, label='Selected Point')
    for i, idx in enumerate(point_indices):
        x, y = surface_x_coord[idx], surface_y_coord[idx]
        if i == 0:
            ax1.plot(x, y, 'go', markersize=8, label=f'Top {N} closest')
        else:
            ax1.plot(x, y, 'go', markersize=8)
        ax1.text(x, y, str(i+1), fontsize=10, ha='center', va='center', 
                color='k', fontweight='bold',
                bbox=dict(boxstyle='circle,pad=0.3', facecolor='white', edgecolor='green', alpha=0.8))
    ax1.legend(loc='best', fontsize=9)
    ax1.grid(True, alpha=0.3)
    
    # Subplot (0,1): Head time series for top N closest points
    ax2 = fig.add_subplot(gs[0, 1])
    colors = plt.cm.tab10(np.linspace(0, 1, N))
    for i, (idx, head_data) in enumerate(zip(point_indices, point_heads)):
        if i == 0:
            ax2.plot(surface_times, head_data, color=colors[i], linewidth=2, 
                    label=f'Closest (idx={idx}, dist={point_distances[i]:.2f}m)')
        else:
            ax2.plot(surface_times, head_data, color=colors[i], linewidth=1.5, linestyle='--',
                    label=f'#{i+1} (idx={idx}, dist={point_distances[i]:.2f}m)')
    ax2.set_xlabel('Time [days]', fontsize=11)
    ax2.set_ylabel('Head [m]', fontsize=11)
    ax2.set_title(f'Head time series for top {N} closest points', fontsize=12, fontweight='bold')
    ax2.legend(loc='best', fontsize=8)
    ax2.grid(True, alpha=0.3)
    
    # Subplot (1,0): Ponded water depth time series
    ax3 = fig.add_subplot(gs[1, 0])
    for i, (idx, pwdep_data) in enumerate(zip(point_indices, point_pwdeps)):
        if i == 0:
            ax3.plot(surface_times, pwdep_data, color=colors[i], linewidth=2, 
                    label=f'Closest (idx={idx}, dist={point_distances[i]:.2f}m)')
        else:
            ax3.plot(surface_times, pwdep_data, color=colors[i], linewidth=1.5, linestyle='--',
                    label=f'#{i+1} (idx={idx}, dist={point_distances[i]:.2f}m)')
    ax3.set_xlabel('Time [days]', fontsize=11)
    ax3.set_ylabel('Ponded water depth [m]', fontsize=11)
    ax3.set_title(f'Ponded water depth time series for top {N} closest points', fontsize=12, fontweight='bold')
    ax3.legend(loc='best', fontsize=8)
    ax3.grid(True, alpha=0.3)
    
    # Subplot (1,1): Scatter plot - 2nd to 7th vs 1st closest
    ax4 = fig.add_subplot(gs[1, 1])
    closest_pwdep = point_pwdeps[0]
    for i in range(1, N):
        ax4.scatter(closest_pwdep, point_pwdeps[i], color=colors[i], alpha=0.5, s=10,
                  label=f'#{i+1} (idx={point_indices[i]}, dist={point_distances[i]:.2f}m)')
    
    # Add 1:1 reference line
    min_val = min(closest_pwdep.min(), min([ep.min() for ep in point_pwdeps[1:]]))
    max_val = max(closest_pwdep.max(), max([ep.max() for ep in point_pwdeps[1:]]))
    ax4.plot([min_val, max_val], [min_val, max_val], 'k--', linewidth=1, label='1:1 line')
    
    ax4.set_xlabel(f'Closest point ponded depth [m] (idx={point_indices[0]})', fontsize=11)
    ax4.set_ylabel('Other points ponded depth [m]', fontsize=11)
    ax4.set_title(f'Ponded water depth: 2nd-7th closest vs 1st closest', fontsize=12, fontweight='bold')
    ax4.legend(loc='best', fontsize=8)
    ax4.grid(True, alpha=0.3)
    ax4.set_aspect('equal')
    
    # Overall title
    fig.suptitle(f'ATS 3D Results for {cur_point["grid_id"]} endpoint P{cur_point["point_id"]} ({cur_point["endpoint_type"]})\n' + 
                 f'Source sites: {cur_point["source_site_ids"]} | Location: ({point_coords[0]:.2f}, {point_coords[1]:.2f})', 
                 fontsize=14, fontweight='bold', y=0.995)
    
    plt.tight_layout()
    
    # Save figure
    output_filename = f"{cur_point['grid_id']}_endpoint_P{cur_point['point_id']:02d}_ATS3D_analysis.png"
    output_path = os.path.join(output_dir, output_filename)
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    print(f"  Saved: {output_filename}")
    
    plt.close(fig)  # Close figure to free memory

print("\n" + "="*80)
print(f"ALL {len(df_points)} PLOTS COMPLETED AND SAVED")
print(f"Output directory: {output_dir}")
print("="*80)